In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

df = pd.read_csv('/content/data_preprocessed (1).csv')
df = df.dropna(subset=['teks_processed', 'sentimen'])

X = df['teks_processed']
y = df['sentimen']

# Split SEBELUM vektorisasi (penting!)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

tfidf = TfidfVectorizer(max_features=500)
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf  = tfidf.transform(X_test)

print('Shape train:', X_train_tfidf.shape)
print('Shape test :', X_test_tfidf.shape)

Shape train: (837, 500)
Shape test : (210, 500)


In [2]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix

nb_model = MultinomialNB()
nb_model.fit(X_train_tfidf, y_train)

pred_nb = nb_model.predict(X_test_tfidf)

print('=== NAIVE BAYES ===')
print(f'Akurasi: {accuracy_score(y_test, pred_nb):.4f}')
print('\nClassification Report:')
print(classification_report(y_test, pred_nb))
print('\nConfusion Matrix:')
print(confusion_matrix(y_test, pred_nb))


=== NAIVE BAYES ===
Akurasi: 0.8524

Classification Report:
              precision    recall  f1-score   support

     negatif       0.81      0.98      0.89       125
     positif       0.97      0.66      0.78        85

    accuracy                           0.85       210
   macro avg       0.89      0.82      0.84       210
weighted avg       0.87      0.85      0.85       210


Confusion Matrix:
[[123   2]
 [ 29  56]]


In [3]:
from sklearn.svm import LinearSVC

svm_model = LinearSVC(C=1.0, random_state=42, max_iter=2000)
svm_model.fit(X_train_tfidf, y_train)

pred_svm = svm_model.predict(X_test_tfidf)

print('=== SVM ===')
print(f'Akurasi: {accuracy_score(y_test, pred_svm):.4f}')
print('\nClassification Report:')
print(classification_report(y_test, pred_svm))
print('\nConfusion Matrix:')
print(confusion_matrix(y_test, pred_svm))


=== SVM ===
Akurasi: 0.9333

Classification Report:
              precision    recall  f1-score   support

     negatif       0.94      0.95      0.94       125
     positif       0.93      0.91      0.92        85

    accuracy                           0.93       210
   macro avg       0.93      0.93      0.93       210
weighted avg       0.93      0.93      0.93       210


Confusion Matrix:
[[119   6]
 [  8  77]]


In [4]:
import joblib

joblib.dump(svm_model, 'svm_sentiment.pkl')
joblib.dump(tfidf, 'tfidf_vectorizer.pkl')

print('Model dan vectorizer berhasil disimpan.')


Model dan vectorizer berhasil disimpan.


In [5]:
import re

def preprocess_full(text):
    text = text.lower()
    text = re.sub(r'[^a-z\s]', '', text)
    return text

review_baru = [
    "Produknya sangat bagus, pengiriman cepat dan packing rapi",
    "Kecewa banget, barang rusak tidak sesuai foto",
    "Lumayan lah, sesuai harga"
]

review_clean = [preprocess_full(r) for r in review_baru]   # fungsi dari Tugas 3
review_vec   = tfidf.transform(review_clean)

print('Prediksi Naive Bayes:', nb_model.predict(review_vec))
print('Prediksi SVM        :', svm_model.predict(review_vec))


Prediksi Naive Bayes: ['positif' 'negatif' 'positif']
Prediksi SVM        : ['positif' 'negatif' 'positif']
